# 02 — PCA and K-Means Customer Clustering

**Owner: Member B**

This notebook loads the splits produced by `01_data_preprocessing.ipynb`, fits preprocessing and clustering on the training set only, evaluates candidate `k` values with the silhouette coefficient, creates cluster profiles, and exports cluster labels plus centroid-distance features.

- Input: `train_70.csv`, `val_15.csv`, `test_15.csv`.
- Features used for clustering: `Recency`, `Frequency`, `Monetary` only.
- `customer_id` and `Repurchase_Label` are never used to fit the scaler, PCA, or K-Means.
- The training-fitted scaler, PCA, and K-Means are reused for validation/test to avoid leakage.

## 1. Imports, paths, and input validation

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path(".")
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

FEATURES = ["Recency", "Frequency", "Monetary"]
TARGET = "Repurchase_Label"
FILES = {"train":"train_70.csv", "val":"val_15.csv", "test":"test_15.csv"}

def read_split(path):
    if not (ROOT / path).exists():
        raise FileNotFoundError(f"{path} not found. Run 01_data_preprocessing.ipynb first.")
    df = pd.read_csv(ROOT / path)
    required = ["customer_id", *FEATURES, TARGET]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")
    if df["customer_id"].duplicated().any():
        raise ValueError(f"Duplicate customer_id in {path}")
    return df

train, val, test = [read_split(FILES[k]) for k in ("train","val","test")]
for name, df in [("train",train),("val",val),("test",test)]:
    print(name, df.shape)
print("Features:", FEATURES)
display(train.head())

## 2. Fit scaler and PCA on training data only

In [ ]:
X_train_raw = train[FEATURES].apply(pd.to_numeric, errors="coerce")
X_val_raw = val[FEATURES].apply(pd.to_numeric, errors="coerce")
X_test_raw = test[FEATURES].apply(pd.to_numeric, errors="coerce")

# Imputation and scaling are fitted only on training data.
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train_raw)
X_val_imp = imputer.transform(X_val_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

pca = PCA(n_components=2, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("PCA explained variance ratio:", np.round(pca.explained_variance_ratio_, 4))
print("Total variance explained by 2 PCs:", round(pca.explained_variance_ratio_.sum(), 4))

## 3. Select the number of clusters using silhouette score

In [ ]:
# K-Means is fitted on training PCA coordinates only.
# Silhouette score is computed on training data; validation/test are not used to choose k.
max_k = min(7, len(train) - 1)
K_RANGE = range(2, max_k + 1)
silhouette_scores = {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_train_pca)
    silhouette_scores[k] = silhouette_score(X_train_pca, labels)

scores_df = pd.DataFrame({
    "k": list(silhouette_scores.keys()),
    "silhouette_score": list(silhouette_scores.values())
})
display(scores_df)

plt.figure(figsize=(7,4))
plt.plot(scores_df["k"], scores_df["silhouette_score"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("K-Means: silhouette score by k (training set)")
plt.xticks(list(K_RANGE))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "silhouette_vs_k.png", dpi=160)
plt.show()
plt.close()

BEST_K = int(scores_df.loc[scores_df["silhouette_score"].idxmax(), "k"])
print(f"Selected k={BEST_K} based on the training-set silhouette score.")
print("If the course report fixes k=3, verify whether the best-k result agrees before finalizing the report.")

## 4. Fit final K-Means and transform all splits

In [ ]:
kmeans = KMeans(n_clusters=BEST_K, random_state=SEED, n_init=10)
train_cluster = kmeans.fit_predict(X_train_pca)
val_cluster = kmeans.predict(X_val_pca)
test_cluster = kmeans.predict(X_test_pca)

# Distances to the same training-fitted centroids, for the optional continuous distance features.
train_dist = kmeans.transform(X_train_pca)
val_dist = kmeans.transform(X_val_pca)
test_dist = kmeans.transform(X_test_pca)

def make_assignment_frame(df, labels, distances, split_name):
    out = pd.DataFrame({
        "customer_id": df["customer_id"].to_numpy(),
        "cluster": labels.astype(int),
        "split": split_name
    })
    for j in range(BEST_K):
        out[f"distance_cluster_{j}"] = distances[:, j]
    return out

assignments = pd.concat([
    make_assignment_frame(train, train_cluster, train_dist, "train"),
    make_assignment_frame(val, val_cluster, val_dist, "val"),
    make_assignment_frame(test, test_cluster, test_dist, "test"),
], ignore_index=True)

assignments.to_csv(OUTPUT_DIR / "cluster_assignments.csv", index=False)
assignments.to_csv(ROOT / "cluster_assignments.csv", index=False)

# Export fitted objects for reuse and reproducibility.
joblib.dump({
    "features": FEATURES,
    "imputer": imputer,
    "scaler": scaler,
    "pca": pca,
    "kmeans": kmeans,
    "best_k": BEST_K,
    "seed": SEED
}, OUTPUT_DIR / "clustering_pipeline.joblib")

print("Cluster assignment file saved:", OUTPUT_DIR / "cluster_assignments.csv")
display(assignments.head())

## 5. Visualize clusters and summarize business profiles

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(X_train_pca[:,0], X_train_pca[:,1], c=train_cluster, cmap="tab10", alpha=0.65, s=18)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"Training customers clustered by K-Means (k={BEST_K})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "clusters_pca2d.png", dpi=160)
plt.show()
plt.close()

profile_data = train[FEATURES + [TARGET]].copy()
profile_data["cluster"] = train_cluster
cluster_profile = profile_data.groupby("cluster").agg(
    customer_count=("cluster", "size"),
    repurchase_rate=(TARGET, "mean"),
    avg_recency=("Recency", "mean"),
    avg_frequency=("Frequency", "mean"),
    avg_monetary=("Monetary", "mean")
).round(3).reset_index()

cluster_profile.to_csv(OUTPUT_DIR / "cluster_profile.csv", index=False)
display(cluster_profile)

# Cluster profiles are descriptive; target labels are not used to create clusters.
plt.figure(figsize=(7,4))
sns.barplot(data=cluster_profile, x="cluster", y="repurchase_rate")
plt.ylabel("Repurchase rate")
plt.title("Repurchase rate by training-set cluster")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cluster_repurchase_rate.png", dpi=160)
plt.show()
plt.close()

## 6. Quality checks and handoff to Member B / C

In [ ]:
# Every customer should have exactly one assignment; each split must match its source IDs.
assert assignments["customer_id"].is_unique, "A customer appears more than once in cluster_assignments.csv"
for split_name, df in [("train",train),("val",val),("test",test)]:
    assigned = set(assignments.loc[assignments["split"] == split_name, "customer_id"])
    expected = set(df["customer_id"])
    assert assigned == expected, f"Assignment mismatch for {split_name}"

print("Checks passed.")
print("Best k:", BEST_K)
print("Files written:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name)
print("\nHandoff: baseline notebook can merge cluster_assignments.csv on customer_id.")
print("C can reuse distance_cluster_* columns as continuous centroid-distance features.")

## Outputs

- `output/silhouette_vs_k.png`
- `output/clusters_pca2d.png`
- `output/cluster_profile.csv`
- `output/cluster_assignments.csv` (customer ID, cluster label, centroid distances, split)
- `output/clustering_pipeline.joblib`

**Important:** the silhouette-selected `BEST_K` may differ from 3. The final report must use the actual result or explicitly justify retaining `k=3`; do not silently report a fixed value that the run did not support.